In [2]:
import numpy as np
import scipy.signal as signal

class AMDemodulator:
    """
    A class to demodulate Amplitude Modulation (AM) signals from complex I/Q data.

    :param fs_rf: The sampling rate of the input RF signal in Hz.
    :type fs_rf: float
    :param fs_audio: The target sampling rate for the output audio in Hz.
    :type fs_audio: float
    """

    def __init__(self, fs_rf, fs_audio):
        self.fs_rf = fs_rf
        self.fs_audio = fs_audio

    def demodulate(self, iq_data):
        """
        Process raw I/Q data to extract normalized audio.

        Pipeline:
        1. Envelope Detection (Magnitude).
        2. DC Offset Removal.
        3. Decimation/Resampling (Anti-aliased).
        4. Low-Pass Filtering.
        5. Normalization.

        :param iq_data: The input complex I/Q samples.
        :type iq_data: numpy.ndarray
        :return: The demodulated, normalized audio samples.
        :rtype: numpy.ndarray
        """
        # 1. Envelope Detection
        envelope = np.abs(iq_data)

        # 2. DC Offset Removal
        envelope_ac = envelope - np.mean(envelope)

        # 3. Decimation / Resampling (includes Anti-Aliasing)
        # Calculate number of samples for target rate
        num_samples = int(len(envelope_ac) * self.fs_audio / self.fs_rf)
        audio_resampled = signal.resample(envelope_ac, num_samples)

        # 4. Low-Pass Filter (Final Audio Cleanup, cut at 5kHz)
        nyquist = 0.5 * self.fs_audio
        sos = signal.butter(4, 5000 / nyquist, btype='low', output='sos')
        audio_filtered = signal.sosfilt(sos, audio_resampled)

        # 5. Output Normalization
        max_val = np.max(np.abs(audio_filtered))
        if max_val > 0:
            return audio_filtered / max_val
        return audio_filtered

if __name__ == "__main__":
    # Settings
    fs_rf = 1e6           # 1 MSps
    fs_audio = 48000      # 48 kHz
    duration = 0.1        # seconds
    tone_freq = 1000      # 1 kHz test tone

    # Generate Synthetic Data
    t = np.arange(0, duration, 1/fs_rf)
    audio_signal = np.cos(2 * np.pi * tone_freq * t)
    
    # Create AM Baseband Signal: (1 + m*signal) * e^(j*0) -> Real envelope in IQ
    modulation_index = 0.5
    # Adding small noise + random phase rotation to simulate real IQ capture
    iq_signal = (1 + modulation_index * audio_signal) * np.exp(1j * np.pi/4) 
    
    # Process
    demod = AMDemodulator(fs_rf, fs_audio)
    result = demod.demodulate(iq_signal)

    # Verification
    print(f"Input Samples: {len(iq_signal)}")
    print(f"Output Samples: {len(result)}")
    print(f"Output RMS: {np.sqrt(np.mean(result**2)):.4f}")
    print("Demodulation pipeline execution successful.")

Input Samples: 100001
Output Samples: 4800
Output RMS: 0.7009
Demodulation pipeline execution successful.
